# Diabetes Risk Prediction

## 02 - Data Preprocessing

### Objectives

- Convert variables to appropriate data types
- Create clinically meaningful categorical variables
- Assess data quality
- Save the processed dataset

### Input

`data/raw/diabetes_raw.parquet`

### Output

`data/processed/diabetes_processed.parquet`

**Note:** Parquet only preserves pandas' `category` dtype for columns whose values are strings — for
numeric-coded columns (like `HighBP` = 0/1) it silently reverts them to `int64` on save. The fix below
is one line: cast to `str` before casting to `category`, so the values Parquet sees are strings.


In [1]:
# ===============================================================
# Import Libraries
# ===============================================================

from pathlib import Path

import numpy as np
import pandas as pd

import pyarrow


In [2]:
# ===============================================================
# Project Configuration
# ===============================================================

def find_project_root(start: Path | None = None) -> Path:
    '''Find the project root by locating the 'notebooks' folder, so this
    works no matter whether the kernel's working directory is the notebook's
    own folder or the workspace root (these can differ between notebooks in
    VS Code and cause confusing FileNotFoundErrors).'''
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if candidate.name == "notebooks":
            return candidate.parent
        if (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find project root above {current}")

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = RAW_DIR / "diabetes_raw.parquet"
PROCESSED_DATA_PATH = PROCESSED_DIR / "diabetes_processed.parquet"


In [3]:
# ===============================================================
# Load Raw Dataset
# ===============================================================

df = pd.read_parquet(RAW_DATA_PATH)

print(f"Observations : {df.shape[0]}")
print(f"Variables    : {df.shape[1]}")

df.head()


Observations : 253680
Variables    : 22


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


In [11]:
# ===============================================================
# Initial Inspection
# ===============================================================

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229474 entries, 0 to 229473
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   HighBP                229474 non-null  category
 1   HighChol              229474 non-null  category
 2   CholCheck             229474 non-null  category
 3   BMI                   229474 non-null  int64   
 4   Smoker                229474 non-null  category
 5   Stroke                229474 non-null  category
 6   HeartDiseaseorAttack  229474 non-null  category
 7   PhysActivity          229474 non-null  category
 8   Fruits                229474 non-null  category
 9   Veggies               229474 non-null  category
 10  HvyAlcoholConsump     229474 non-null  category
 11  AnyHealthcare         229474 non-null  category
 12  NoDocbcCost           229474 non-null  category
 13  GenHlth               229474 non-null  category
 14  MentHlth              229474 non-nul

### Convert Data Types

Cast the coded survey/indicator columns to `category`. They're cast to `str` first so the dtype
survives being saved to and re-read from Parquet (see note above).


In [5]:
# ===============================================================
# Convert Data Types
# ===============================================================

categorical_columns = [
    "Diabetes_binary", "HighBP", "HighChol", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "DiffWalk", "Sex", "Education", "Income", "CholCheck", "Age"
]

df[categorical_columns] = df[categorical_columns].astype(str).astype("category")

df.dtypes


HighBP                  category
HighChol                category
CholCheck               category
BMI                        int64
Smoker                  category
Stroke                  category
HeartDiseaseorAttack    category
PhysActivity            category
Fruits                  category
Veggies                 category
HvyAlcoholConsump       category
AnyHealthcare           category
NoDocbcCost             category
GenHlth                 category
MentHlth                   int64
PhysHlth                   int64
DiffWalk                category
Sex                     category
Age                     category
Education               category
Income                  category
Diabetes_binary         category
dtype: object

In [6]:
# ===============================================================
# Create BMI Categories
# ===============================================================

df["BMI_cat"] = pd.cut(
    df["BMI"],
    bins=[-np.inf, 18.5, 25, 30, np.inf],
    labels=["Underweight", "Normal weight", "Overweight", "Obese"],
)


In [7]:
# ===============================================================
# Create Mental & Physical Health Categories
# ===============================================================

day_bins = [-1, 0, 10, 20, 30]
day_labels = ["0 days", "1-10 days", "11-20 days", "21-30 days"]

df["MentHlth_cat"] = pd.cut(df["MentHlth"], bins=day_bins, labels=day_labels)
df["PhysHlth_cat"] = pd.cut(df["PhysHlth"], bins=day_bins, labels=day_labels)


In [8]:
# ===============================================================
# Data Quality Assessment
# ===============================================================

print(f"Observations : {df.shape[0]}")
print(f"Variables    : {df.shape[1]}")
print(f"Duplicates   : {df.duplicated().sum()}")

print()
print("Missing Values")
display(df.isnull().sum())

print()
print("Data Types")
display(df.dtypes)


Observations : 253680
Variables    : 25
Duplicates   : 24206

Missing Values


HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
Diabetes_binary         0
BMI_cat                 0
MentHlth_cat            0
PhysHlth_cat            0
dtype: int64


Data Types


HighBP                  category
HighChol                category
CholCheck               category
BMI                        int64
Smoker                  category
Stroke                  category
HeartDiseaseorAttack    category
PhysActivity            category
Fruits                  category
Veggies                 category
HvyAlcoholConsump       category
AnyHealthcare           category
NoDocbcCost             category
GenHlth                 category
MentHlth                   int64
PhysHlth                   int64
DiffWalk                category
Sex                     category
Age                     category
Education               category
Income                  category
Diabetes_binary         category
BMI_cat                 category
MentHlth_cat            category
PhysHlth_cat            category
dtype: object

In [9]:
# ===============================================================
# Remove Duplicates & Save
# ===============================================================

df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")

df.to_parquet(PROCESSED_DATA_PATH, index=False)
print(f"Saved to {PROCESSED_DATA_PATH}")


Shape after dropping duplicates: (229474, 25)
Saved to C:\Users\saifu\OneDrive\Documents\Python_Demo\Diabetes-Risk-Prediction\data\processed\diabetes_processed.parquet


In [10]:
# ===============================================================
# Verify Saved Dataset
# ===============================================================

df_check = pd.read_parquet(PROCESSED_DATA_PATH)

print(f"Observations : {df_check.shape[0]}")
print(f"Variables    : {df_check.shape[1]}")
display(df_check.head())

print()
print("Data Types")
display(df_check.dtypes)


Observations : 229474
Variables    : 25


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary,BMI_cat,MentHlth_cat,PhysHlth_cat
0,1,1,1,40,1,0,0,0,0,1,...,15,1,0,9,4,3,0,Obese,11-20 days,11-20 days
1,0,0,0,25,1,0,0,1,0,0,...,0,0,0,7,6,1,0,Normal weight,0 days,0 days
2,1,1,1,28,0,0,0,0,1,0,...,30,1,0,9,4,8,0,Overweight,21-30 days,21-30 days
3,1,0,1,27,0,0,0,1,1,1,...,0,0,0,11,3,6,0,Overweight,0 days,0 days
4,1,1,1,24,0,0,0,1,1,1,...,0,0,0,11,5,4,0,Normal weight,1-10 days,0 days



Data Types


HighBP                  category
HighChol                category
CholCheck               category
BMI                        int64
Smoker                  category
Stroke                  category
HeartDiseaseorAttack    category
PhysActivity            category
Fruits                  category
Veggies                 category
HvyAlcoholConsump       category
AnyHealthcare           category
NoDocbcCost             category
GenHlth                 category
MentHlth                   int64
PhysHlth                   int64
DiffWalk                category
Sex                     category
Age                     category
Education               category
Income                  category
Diabetes_binary         category
BMI_cat                 category
MentHlth_cat            category
PhysHlth_cat            category
dtype: object